# 폐루프 탐색 실습

**Closed-loop Discovery · 순환 탐색**

후보 제안·실험 또는 계산·결과 분석·모델 갱신을 반복하는 탐색 방식.

소재 분야에서 이해하기: 새 실험 결과에 따라 다음 조성 제안을 바꾼다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [자율 소재 실험실 A-Lab 연구](https://www.nature.com/articles/s41586-023-06734-w)

## 1. 제안 → 실험 → 학습을 반복

루프를 돌릴 때마다 모델이 갱신되어 제안이 바뀌는 것을 확인합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm

def run_experiment(x):
    """가상 합성 결과(수율 %). 측정 잡음이 있습니다."""
    value = 85 * np.exp(-((x[0] - 0.65) ** 2 + (x[1] - 0.35) ** 2) / 0.06)
    return value + np.random.default_rng(int(1e6 * (x[0] + x[1]))).normal(0, 1.5)

grid_a, grid_b = np.meshgrid(np.linspace(0, 1, 60), np.linspace(0, 1, 60))
candidate_grid = np.column_stack([grid_a.ravel(), grid_b.ravel()])
print('탐색 공간 %d점, 루프 1회에 실험 1건' % len(candidate_grid))

In [ ]:
seen_x = [[0.1, 0.9], [0.9, 0.1], [0.5, 0.5]]
seen_y = [run_experiment(np.array(point)) for point in seen_x]
history = [max(seen_y)]
snapshots = {}
for cycle in range(1, 16):
    model = GaussianProcessRegressor(kernel=ConstantKernel(50.0) * RBF([0.2, 0.2]),
                                     normalize_y=True, alpha=2.0, random_state=0)
    model.fit(np.array(seen_x), seen_y)
    mean, std = model.predict(candidate_grid, return_std=True)
    best = max(seen_y)
    z = (mean - best) / np.maximum(std, 1e-9)
    acquisition = (mean - best) * norm.cdf(z) + std * norm.pdf(z)
    pick = candidate_grid[int(np.argmax(acquisition))]
    seen_x.append(list(pick)); seen_y.append(run_experiment(pick))
    history.append(max(seen_y))
    if cycle in (1, 5, 15):
        snapshots[cycle] = (mean.copy(), np.array(seen_x).copy())
    print('cycle %2d 제안 (%.2f, %.2f) -> 수율 %.1f%% (최고 %.1f%%)'
          % (cycle, pick[0], pick[1], seen_y[-1], history[-1]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for axis, cycle in zip(axes, sorted(snapshots)):
    mean, points = snapshots[cycle]
    axis.contourf(grid_a, grid_b, mean.reshape(grid_a.shape), levels=20, cmap='viridis')
    axis.scatter(points[:, 0], points[:, 1], c='red', s=18)
    axis.scatter([0.65], [0.35], marker='*', c='white', s=140)
    axis.set_title('after cycle %d' % cycle); axis.set_xlabel('condition a')
axes[0].set_ylabel('condition b')
plt.tight_layout(); plt.show()
plt.plot(history, 'o-'); plt.xlabel('cycle'); plt.ylabel('best yield so far (%)'); plt.show()
print('흰 별이 참 최적점입니다. 루프가 돌면서 실험이 그 주변으로 모입니다.')

## 2. 해석

폐루프의 성능은 모델·획득 함수·실험 잡음이 함께 결정합니다. 측정 잡음을 모델에 알려주지
않으면(alpha 를 너무 작게 두면) 잡음이 큰 한 점을 최적점으로 착각해 루프가 갇힐 수 있습니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#closed-loop)을 여세요.